In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install -q transformers datasets evaluate jiwer accelerate albumentations opencv-python-headless

In [3]:
import torch
import numpy as np
import albumentations as A
from PIL import Image
from datasets import load_dataset
from transformers import (
    TrOCRProcessor, 
    VisionEncoderDecoderModel, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments,
    default_data_collator
)
import evaluate

print(f"⚙️ GPU Available: {torch.cuda.is_available()}")

⚙️ GPU Available: True


In [4]:
print("⚙️ Initializing StyleScript Phase 4 Stochastic Augmentation Pipeline...")

# Translating the paper's mathematical constraints into an OpenCV/Albumentations pipeline
stylescript_transform = A.Compose([
    # Eq 11: Geometric Transform (Rotation between -3 and 3 degrees)
    A.Rotate(limit=3, p=0.8, border_mode=0, value=(255, 255, 255)),
    
    # Eq 12: Perspective Transform (Mimicking scanning distortion)
    A.Perspective(scale=(0.01, 0.03), p=0.5, fit_output=True),
    
    # Eq 13: Noise Addition (Gaussian noise injection)
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.6),
    
    # Eq 14: Photometric Transform (Brightness/Contrast factor ~ U(0.95, 1.05))
    A.RandomBrightnessContrast(brightness_limit=0.05, contrast_limit=0.05, p=0.8)
])
print("✅ StyleScript Pipeline Ready!")

⚙️ Initializing StyleScript Phase 4 Stochastic Augmentation Pipeline...
✅ StyleScript Pipeline Ready!


/tmp/ipykernel_373/2048878027.py:6: UserWarning: Argument(s) 'value' are not valid for transform Rotate
  A.Rotate(limit=3, p=0.8, border_mode=0, value=(255, 255, 255)),
/tmp/ipykernel_373/2048878027.py:12: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.6),


In [5]:
print("📥 Downloading IAM Sentences Dataset...")
dataset = load_dataset("alpayariyak/IAM_Sentences", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print("📥 Loading Base TrOCR Model...")
checkpoint = "microsoft/trocr-base-handwritten"
processor = TrOCRProcessor.from_pretrained(checkpoint)
model = VisionEncoderDecoderModel.from_pretrained(checkpoint)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

📥 Downloading IAM Sentences Dataset...


📥 Loading Base TrOCR Model...


The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
print("🔒 Freezing the Vision Encoder...")

for param in model.encoder.parameters():
    param.requires_grad = False
    
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"We are only training {((trainable_params/total_params)*100):.2f}% of the model!")

🔒 Freezing the Vision Encoder...
Total parameters: 333,921,792
Trainable parameters: 247,268,352
We are only training 74.05% of the model!


In [7]:
def preprocess_data(batch):
    processed_images = []
    
    for image in batch["image"]:
        img_array = np.array(image.convert("RGB"))
        
        # APPLY THE STYLESCRIPT PAPER MATH
        augmented_array = stylescript_transform(image=img_array)["image"]
        processed_images.append(Image.fromarray(augmented_array))
        
    texts = batch["text"]

    pixel_values = processor(processed_images, return_tensors="pt").pixel_values
    
    labels = processor.tokenizer(
        texts, padding="max_length", max_length=128, truncation=True
    ).input_ids

    labels = [
        [-100 if token == processor.tokenizer.pad_token_id else token for token in label]
        for label in labels
    ]

    return {"pixel_values": pixel_values, "labels": labels}

print("🔄 Processing Dataset (Processing in small batches to prevent RAM crash)...")
train_dataset = train_dataset.map(
    preprocess_data, 
    batched=True, 
    batch_size=64, # 🚨 THE FIX: Processes 64 images at a time instead of 1000!
    remove_columns=train_dataset.column_names
)
eval_dataset = eval_dataset.map(
    preprocess_data, 
    batched=True, 
    batch_size=64, 
    remove_columns=eval_dataset.column_names
)

train_dataset.set_format(type="torch")
eval_dataset.set_format(type="torch")
print("✅ Dataset mapped successfully without memory overflow!")

🔄 Processing Dataset (Processing in small batches to prevent RAM crash)...


Map:   0%|          | 0/5096 [00:00<?, ? examples/s]

Map:   0%|          | 0/567 [00:00<?, ? examples/s]

✅ Dataset mapped successfully without memory overflow!


In [8]:
import evaluate  # 🚨 THE FIX: Re-importing the library directly in this cell

cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}

training_args = Seq2SeqTrainingArguments(
    output_dir="./stylescript-trocr",
    predict_with_generate=True,
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=True, 
    learning_rate=4e-5, 
    num_train_epochs=15, 
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    processing_class=processor.image_processor, 
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
)

In [9]:
print("🚀 Starting Fine-Tuning with StyleScript Augmentation on GPU...")
# This is the line that actually starts the 15 epochs
trainer.train()

print("💾 Saving the final model weights locally to Kaggle...")
# This saves the trained neural network brain
trainer.save_model("./stylescript-trocr-final")
# This saves the image processor settings
processor.save_pretrained("./stylescript-trocr-final")

print("✅ Training Complete and Saved Locally! You are safe to upload now.")

🚀 Starting Fine-Tuning with StyleScript Augmentation on GPU...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Cer
1,9.404253,4.551337,0.720032
2,8.169584,4.211088,0.719947
3,7.377084,4.108576,0.716350
4,6.384459,3.993236,0.709584
5,5.822284,3.891787,0.697074
6,5.473961,3.853448,0.695403
7,4.838040,3.904722,0.687853
8,4.295216,3.910014,0.674558
9,3.542848,3.943805,0.682620
10,3.344891,3.954602,0.675376


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Saving the final model weights locally to Kaggle...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training Complete and Saved Locally! You are safe to upload now.
